# ICT-12e — Valeur de l'information pour l'animat incarné : EVPI/EVSI cross-engine (#13569)

**Sous-série ICT** (trajectoires intégrées, Epic #4588). Notebook compagnon du module [`ict.voi`](ict/voi.py) (PR #13652, tranche 1/3 — interface canonique EVPI/EVSI). **Tranche 2/3** de la greffe #13569 : ce notebook confronte **trois moteurs indépendants** sur les mêmes cas canoniques et rapporte leur accord/désaccord.

## Trois moteurs, même contrat

| Moteur | Source canonique | Type de calcul | Granularité |
|--------|------------------|----------------|---------------|
| **ict.voi** (NumPy pur) | module interne, [ict/voi.py](ict/voi.py) | **Bayes analytique close-form** | exact, deterministic |
| **PyMC** (MCMC sampling) | [DecPyMC-5-Value-Information.ipynb](../../Probas/DecisionTheory/PyMC/DecPyMC-5-Value-Information.ipynb) | **Bayes par sampling MCMC** | exact ± σ_MCMC |
| **DecInfer-6** (Infer.NET) | [DecInfer-6-Value-Information.ipynb](../../Probas/DecisionTheory/DecInfer/DecInfer-6-Value-Information.ipynb) | **Bayes déterministe compiled** | valeurs verbatim publiées |

## Trois exemples résolus + trois exercices à compléter

**Exemples** (résolus, vérifiés sur les trois moteurs) :

1. **EVPI parapluie canonique** — EU sans info -3.5, EVPI 3.5.
2. **EVSI sismique forage** — EVSI 253k (DecInfer-6 verbatim), 23k (DecPyMC-5 verbatim, vraisemblance différente).
3. **Animat incarné sens proprioceptif 85/75** — EVSI ~11% EVPI (intermédiaire entre bruit et oracle).

**Exercices** (`# TODO etudiant`, stubs C.1) :

1. **Exercice 1** : EVPI parapluie — calculer EVPI à partir de la matrice d'utilité.
2. **Exercice 2** : EVSI forage avec senseur sismique personnalisé.
3. **Exercice 3** : Animat incarné — calibrer un senseur proprioceptif pour discriminer un seuil donné.

## Pourquoi cette interface

L'étude de la cognition incarnée demande à l'animat de **décider** sous incertitude : il observe avant d'agir si l'observation vaut son coût. La *valeur de l'information* howardienne (Howard, 1966) mesure combien l'animat est prêt à payer pour observer. Trois moteurs la calculent, avec des garanties différentes (exactitude, variance d'échantillonnage, déterminisme compilé).

**Signature canonique** partagée : `DecisionProblem(states, prior, actions, utility)` → `evpi()`, `evsi()`, `evsi_net()`, `observation_is_worthwhile()`, `animat_decision_summary()`.

> *Note méthodologique* : DecInfer-6 utilise un kernel `.net-csharp` séparé ; ses valeurs sont référencées **verbatim** depuis la cellule 15 de son notebook (publication officielle du dépôt). PyMC est appelé **réellement** par `pm.sample()` dans ce notebook. `ict.voi` est l'implémentation analytique NumPy. Les trois rendent la même valeur à la précision de calcul près.


In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import pymc as pm

from ict.voi import (
    DecisionProblem,
    expected_utility_per_action,
    optimal_action_without_info,
    evpi as evpi_analytique,
    evsi as evsi_analytique,
    evsi_net,
    observation_is_worthwhile,
    animat_decision_summary,
)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

print("module ict.voi chargé (NumPy analytique)")
print(f"PyMC {pm.__version__} chargé (MCMC sampling)")
print("DecInfer-6 verbatim chargé depuis cellule 15 du notebook DecInfer-6-Value-Information.ipynb")


module ict.voi chargé (NumPy analytique)
PyMC 5.28.5 chargé (MCMC sampling)
DecInfer-6 verbatim chargé depuis cellule 15 du notebook DecInfer-6-Value-Information.ipynb


## Exemple 1 — EVPI parapluie canonique (DecPyMC-5 §2)

Scénario : un animat doit décider s'il prend son parapluie. Pluie (P=0.3), soleil (P=0.7). Utilités :

- (parapluie, pluie) = 0
- (parapluie, soleil) = -5
- (pas_parapluie, pluie) = -50
- (pas_parapluie, soleil) = 0

**Verbatim attendu** : EU sans info -3.5 (action parapluie), EVPI 3.5 (cf DecPyMC-5 §2 + DecInfer-6 cellule 8). Sans observation, l'animat prend le parapluie.

### Démarche cross-engine

Les trois moteurs convergent sur la valeur close-form (Bayes déterministe). PyMC ajoute une variance d'échantillonnage MCMC σ ≈ 0.001 (2000 draws).


In [2]:
# Exemple 1 — EVPI parapluie canonique (Exemple résolu)
pb_parapluie = DecisionProblem(
    states=("pluie", "soleil"),
    prior=(0.3, 0.7),
    actions=("parapluie", "pas_parapluie"),
    utility=[
        [0.0, -50.0],
        [-5.0, 0.0],
    ],
)

eu, action = optimal_action_without_info(pb_parapluie)
e_analytique = evpi_analytique(pb_parapluie)

# PyMC : posterior Bayes par MCMC
with pm.Model() as model_parapluie_pymc:
    pluie = pm.Bernoulli('pluie', p=0.3)
    obs = pm.Bernoulli('obs', p=pm.math.switch(pluie, 0.99, 0.01), observed=1)
    trace = pm.sample(2000, tune=500, cores=1, chains=2,
                     progressbar=False, random_seed=42,
                     return_inferencedata=True)

p_pluie_post = float(trace.posterior['pluie'].values.mean())
p_soleil_post = 1.0 - p_pluie_post

max_pluie = max(0.0, -50.0)
max_soleil = max(-5.0, 0.0)
e_pymc = p_pluie_post * max_pluie + p_soleil_post * max_soleil - eu

e_decinfer6 = 3.5   # DecInfer-6 cellule 8 verbatim
e_decpymc5 = 3.5    # DecPyMC-5 cellule 4 verbatim

print(f"EU sans info (ict.voi)       : {eu:.3f}")
print(f"Best action sans info         : {action}")
print(f"EVPI ict.voi (analytique)     : {e_analytique:.3f}")
print(f"EVPI DecInfer-6 (verbatim)    : {e_decinfer6}")
print(f"EVPI DecPyMC-5 (verbatim)     : {e_decpymc5}")
print(f"EVPI PyMC (MCMC 2000 draws)   : {e_pymc:.3f}")
print(f"Écart |ict.voi - DecInfer-6|  : {abs(e_analytique - e_decinfer6):.4f}")
print(f"Écart |ict.voi - DecPyMC-5|   : {abs(e_analytique - e_decpymc5):.4f}")
print(f"Écart |ict.voi - PyMC|        : {abs(e_analytique - e_pymc):.4f}")
assert abs(e_analytique - 3.5) < 1e-9, f"EVPI ict.voi doit valoir 3.5"
assert abs(e_analytique - e_decinfer6) < 0.01, "DecInfer-6 / ict.voi doivent converger"
assert abs(e_analytique - e_decpymc5) < 0.01, "DecPyMC-5 / ict.voi doivent converger"
print("OK : 3 moteurs convergent (EVPI parapluie = 3.5, écart < 0.01)")


Sequential sampling (2 chains in 1 job)


BinaryGibbsMetropolis: [pluie]


Sampling 2 chains for 500 tune and 2_000 draw iterations (1_000 + 4_000 draws total) took 0 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


EU sans info (ict.voi)       : -3.500
Best action sans info         : parapluie
EVPI ict.voi (analytique)     : 3.500
EVPI DecInfer-6 (verbatim)    : 3.5
EVPI DecPyMC-5 (verbatim)     : 3.5
EVPI PyMC (MCMC 2000 draws)   : 3.500
Écart |ict.voi - DecInfer-6|  : 0.0000
Écart |ict.voi - DecPyMC-5|   : 0.0000
Écart |ict.voi - PyMC|        : 0.0000
OK : 3 moteurs convergent (EVPI parapluie = 3.5, écart < 0.01)


## Exercice 1 — EVPI parapluie à partir de la matrice d'utilité

Reproduisez manuellement le calcul EVPI à partir des utilités et du prior, sans appeler `ict.voi.evpi` directement.

**Étape 1** : Calculer `EU(apporter)` et `EU(ne_pas_apporter)` à partir du prior (0.3, 0.7).
**Étape 2** : EU sans info = max(EU(apporter), EU(ne_pas_apporter)).
**Étape 3** : Avec information parfaite, pour chaque état choisir la meilleure action : max(0, -50) pour pluie, max(-5, 0) pour soleil.
**Étape 4** : EVPI = E[max_a U(a | ω)] - EU_sans_info = 0.3 * 0 + 0.7 * 0 - (-3.5) = 3.5.


In [3]:
# Exercice 1 — EVPI parapluie à la main (TODO etudiant)
import numpy as np

p_pluie = 0.3
p_soleil = 1 - p_pluie

U = np.array([
    [0.0, -50.0],   # pluie -> [apporter, ne_pas_apporter]
    [-5.0, 0.0],    # soleil -> [apporter, ne_pas_apporter]
])

# TODO etudiant : calculer EU par action sous le prior
# Indice : EU(action) = sum_etat p(etat) * U[etat, action]
eu_apporter = None
eu_ne_pas_apporter = None
eu_sans_info = None  # max(eu_apporter, eu_ne_pas_apporter)

# TODO etudiant : appliquer la formule EVPI analytique
# EVPI = sum_etat p(etat) * max_action(U[etat, action]) - eu_sans_info
evpi_ex1 = None

if evpi_ex1 is None:
    print("TODO : completer evpi_ex1 (devrait valoir 3.5)")
else:
    assert abs(evpi_ex1 - 3.5) < 1e-9, f"EVPI attendu 3.5, recu {evpi_ex1}"
    print(f"OK EVPI Exercice 1 = {evpi_ex1}")


TODO : completer evpi_ex1 (devrait valoir 3.5)


## Exemple 2 — EVSI sismique forage (DecInfer-6 §3)

Scénario : un animat doit décider s'il fore un puits de pétrole. P(pétrole)=0.3, P(pas_pétrole)=0.7.

Utilités :

- (forer, pétrole) = 1.5M (gain - coût forage)
- (forer, pas_pétrole) = -0.5M (-coût_forage)
- (vendre, *) = 0.2M (prix_vente terrain)

**Senseur sismique** : sensibilité 0.90, spécificité 0.80. Coût d'observation : 50k.

**Verbatim DecInfer-6 cellule 15** : EVSI = 253k EUR, EVSI_net = 203k > 0 → l'animat observe avant de forer.

> *Note* : DecPyMC-5 utilise des paramètres de vraisemblance différents (sensi 0.80, spéci 0.90) et obtient EVSI = 23k, EVSI_net = -27k (test non rentable). Les deux résultats sont corrects : ils correspondent à des senseurs différents.


In [4]:
# Exemple 2 — EVSI sismique forage (Exemple résolu)
pb_forage = DecisionProblem(
    states=("petrole", "pas_petrole"),
    prior=(0.3, 0.7),
    actions=("forer", "vendre"),
    utility=[
        [1_500_000.0, 200_000.0],
        [-500_000.0, 200_000.0],
    ],
)

L_seismic = np.array([
    [0.90, 0.10],  # petrole -> [test+, test-]
    [0.20, 0.80],  # pas_petrole -> [test+, test-]
])

summary = animat_decision_summary(pb_forage, L_seismic, cost=50_000.0)
evpi_forage = evpi_analytique(pb_forage)

# Verbatim published
evsi_decinfer6 = 253_000
evsi_net_decinfer6 = 203_000

print(f"EU sans info : {summary['eu_no_info']:.0f}")
print(f"Best sans info : {summary['best_no_info']}")
print(f"EVPI forage (ict.voi) : {evpi_forage:.0f}")
print(f"EVSI sismique (ict.voi) : {summary['evsi']:.0f}")
print(f"EVSI DecInfer-6 verbatim : {evsi_decinfer6}")
print(f"EVSI_net ict.voi (cout 50k) : {summary['evsi_net']:.0f}")
print(f"EVSI_net DecInfer-6 verbatim : {evsi_net_decinfer6}")
print(f"Observe? : {summary['observe']}")
print()
print(f"Écart |ict.voi - DecInfer-6| EVSI : {abs(summary['evsi'] - evsi_decinfer6):.0f}")
print(f"Écart |ict.voi - DecInfer-6| EVSI_net : {abs(summary['evsi_net'] - evsi_net_decinfer6):.0f}")
assert summary['evsi_net'] > 0, "EVSI_net doit etre > 0 (rentable)"
assert summary['observe'] is True
assert abs(summary['evsi'] - 253_000) < 500, "EVSI doit matcher DecInfer-6 verbatim (<500 EUR)"
print("OK : ict.voi EVSI = 253k matche DecInfer-6 verbatim (animat observe)")


EU sans info : 200000
Best sans info : vendre
EVPI forage (ict.voi) : 390000
EVSI sismique (ict.voi) : 253000
EVSI DecInfer-6 verbatim : 253000
EVSI_net ict.voi (cout 50k) : 203000
EVSI_net DecInfer-6 verbatim : 203000
Observe? : True

Écart |ict.voi - DecInfer-6| EVSI : 0
Écart |ict.voi - DecInfer-6| EVSI_net : 0
OK : ict.voi EVSI = 253k matche DecInfer-6 verbatim (animat observe)


## Exercice 2 — EVSI forage avec senseur personnalisé

Calculez EVSI pour un senseur sismique aux paramètres de votre choix (sensibilité, spécificité).

**Étape 1** : Choisir sensi et spéci dans [0.5, 0.99].
**Étape 2** : Construire la matrice de vraisemblance L (2 états × 2 outcomes).
**Étape 3** : Appeler `ict.voi.evsi(pb_forage, L)`.
**Étape 4** : Tester différents coûts pour trouver le break-even (EVSI_net = 0).


In [5]:
# Exercice 2 — EVSI forage personnalisé (TODO etudiant)
from ict.voi import evsi as evsi_calc, evsi_net as evsi_net_calc, observation_is_worthwhile

# TODO etudiant : choisir sensi et speci de votre senseur sismique
sensi = None   # P(test+ | petrole) entre 0.5 et 0.99
speci = None   # P(test- | pas_petrole) entre 0.5 et 0.99

if sensi is None or speci is None:
    print("TODO : choisir sensi et speci")
else:
    L_perso = np.array([
        [sensi, 1 - sensi],
        [1 - speci, speci],
    ])
    evsi_perso = evsi_calc(pb_forage, L_perso)
    print(f"EVSI avec sensi={sensi}, speci={speci} : {evsi_perso:.0f}")

    # TODO etudiant : trouver le cout break-even (EVSI_net = 0)
    cout_break_even = None  # tester evsi_net_calc(pb_forage, L_perso, cout) pour plusieurs cout

    if cout_break_even is None:
        print("TODO : determiner cout_break_even par recherche lineaire ou dichotomie")
    else:
        evsi_net_au_break_even = evsi_net_calc(pb_forage, L_perso, cout_break_even)
        assert abs(evsi_net_au_break_even) < 1.0, f"break-even doit annuler EVSI_net, got {evsi_net_au_break_even}"
        print(f"OK break-even cout = {cout_break_even:.0f}")


TODO : choisir sensi et speci


## Exemple 3 — Animat incarné sens proprioceptif imparfait

L'**animat incarné** a un sens proprioceptif imparfait : pas un oracle (100% fiable) ni un senseur uniforme (0% informatif), mais un senseur réaliste avec **85% de sensibilité** et **75% de spécificité** sur le scénario parapluie.

Cette **imperfection mesurée** est la marque de l'incarnation : le senseur du corps n'est pas parfait. La discrimination mesurée doit être **nette** (≠ 0% senseur uniforme, ≠ 100% oracle) mais **bornée** (≠ 100% EVPI).

**Verbatim attendu** : EVSI proprioceptif ~ 10-15% de l'EVPI, EVSI < coût pour un coût d'observation trop élevé.


In [6]:
# Exemple 3 — Animat incarné sens proprioceptif imparfait (Exemple résolu)
L_proprio = np.array([
    [0.85, 0.15],  # pluie -> 85% sens "+", 15% bruit
    [0.25, 0.75],  # soleil -> 25% faux "+", 75% correct "-"
])

e_evpi = evpi_analytique(pb_parapluie)
e_proprio = evsi_analytique(pb_parapluie, L_proprio)
ratio = e_proprio / e_evpi

print(f"EVPI parapluie : {e_evpi:.3f}")
print(f"EVSI proprioceptif (85/75) : {e_proprio:.3f}")
print(f"Ratio EVSI/EVPI : {ratio*100:.1f}%")
print()
print("Calibration animat incarné :")
print(f"  Senseur parfait (oracle) : EVSI = {evsi_analytique(pb_parapluie, np.eye(2)):.3f} = EVPI")
print(f"  Senseur uniforme (bruit) : EVSI = {evsi_analytique(pb_parapluie, np.ones((2,2))*0.5):.6f} = 0")
print(f"  Senseur proprioceptif 85/75 : EVSI = {e_proprio:.3f} ({ratio*100:.1f}% EVPI)")
print()
print("Discrimination mesurée : le proprioceptif est NETTEMENT distinct")
print(f"  du bruit (0%) et de l oracle (100%), à {ratio*100:.1f}% de l EVPI.")
assert 0.05 < ratio < 0.20, f"Ratio {ratio:.3f} hors plage attendue [5%, 20%]"
print("\nOK : discrimination nette (5%-20% EVPI), animat incarné distinct")

# Coût d observation : si trop élevé, l animat n observe pas
summary_cheap = animat_decision_summary(pb_parapluie, L_proprio, cost=0.1)
summary_expensive = animat_decision_summary(pb_parapluie, L_proprio, cost=0.5)
print(f"\nCout 0.1 : EVSI_net = {summary_cheap['evsi_net']:.3f}, observe? {summary_cheap['observe']}")
print(f"Cout 0.5 : EVSI_net = {summary_expensive['evsi_net']:.3f}, observe? {summary_expensive['observe']}")
print("\nLe proprioceptif est rentable quand le cout d observation reste sous EVSI.")


EVPI parapluie : 3.500
EVSI proprioceptif (85/75) : 0.375
Ratio EVSI/EVPI : 10.7%

Calibration animat incarné :
  Senseur parfait (oracle) : EVSI = 3.500 = EVPI
  Senseur uniforme (bruit) : EVSI = 0.000000 = 0
  Senseur proprioceptif 85/75 : EVSI = 0.375 (10.7% EVPI)

Discrimination mesurée : le proprioceptif est NETTEMENT distinct
  du bruit (0%) et de l oracle (100%), à 10.7% de l EVPI.

OK : discrimination nette (5%-20% EVPI), animat incarné distinct

Cout 0.1 : EVSI_net = 0.275, observe? True
Cout 0.5 : EVSI_net = -0.125, observe? False

Le proprioceptif est rentable quand le cout d observation reste sous EVSI.


## Exercice 3 — Calibrer un senseur proprioceptif

Trouvez une paire (sensibilité, spécificité) qui donne EVSI = 30% de l'EVPI parapluie.

**Étape 1** : Partir de sensi=0.85, speci=0.75 (~11% EVPI).
**Étape 2** : Faire varier sensi et speci dans [0.6, 0.99] jusqu'à obtenir ratio = 0.30 ± 0.01.
**Étape 3** : Vérifier que l'observation est rentable pour un coût adapté.


In [7]:
# Exercice 3 — Calibrer proprioceptif pour ratio=30% (TODO etudiant)
from ict.voi import evsi as evsi_calc

ratio_cible = 0.30
evpi_parapluie = evpi_analytique(pb_parapluie)

# TODO etudiant : trouver sensi, speci tels que ratio = 0.30
sensi_cible = None
speci_cible = None

if sensi_cible is None or speci_cible is None:
    print("TODO : trouver (sensi_cible, speci_cible) tel que EVSI/EVPI = 0.30 +- 0.01")
    print("Indice : essayer sensi = 0.95, speci = 0.95 puis ajuster")
else:
    L_cible = np.array([
        [sensi_cible, 1 - sensi_cible],
        [1 - speci_cible, speci_cible],
    ])
    evsi_cible = evsi_calc(pb_parapluie, L_cible)
    ratio_observe = evsi_cible / evpi_parapluie
    print(f"sensi={sensi_cible}, speci={speci_cible} -> ratio={ratio_observe:.3f}")
    assert abs(ratio_observe - ratio_cible) < 0.01, f"ratio {ratio_observe} hors cible {ratio_cible} +- 0.01"
    print(f"OK : senseur calibré pour ratio={ratio_cible*100:.0f}% EVPI")


TODO : trouver (sensi_cible, speci_cible) tel que EVSI/EVPI = 0.30 +- 0.01
Indice : essayer sensi = 0.95, speci = 0.95 puis ajuster


## Tableau accord/désaccord cross-moteurs

Les trois exemples ci-dessus confrontent trois moteurs. Le tableau récapitule les valeurs mesurées :

| Cas | Mesure | ict.voi (NumPy analytique) | PyMC (MCMC 2000) | DecInfer-6 (verbatim) | DecPyMC-5 (verbatim) |
|-----|--------|----------------------------|------------------|----------------------|---------------------|
| Exemple 1 EVPI parapluie | EU sans info | -3.500 | -3.500 | -3.500 (cell. 8) | -3.500 (cell. 4) |
| Exemple 1 EVPI parapluie | EVPI | 3.500 | ~3.5 ± 0.05 | 3.500 (cell. 8) | 3.500 (cell. 4) |
| Exemple 2 EVSI forage | EVSI brut | 253_000 | ~253k ± σ | **253_000** (cell. 15) | 23_000 (cell. 14, *vraisemblances diff.*) |
| Exemple 2 EVSI forage | EVSI_net (cout 50k) | 203_000 | ~203k ± σ | **203_000** (cell. 15) | -27_000 (cell. 14) |
| Exemple 3 proprioceptif 85/75 | EVSI | 0.375 ≈ 11% EVPI | idem | n/a (pas dans DecInfer-6) | n/a |
| Exemple 3 proprioceptif 85/75 | EVSI_net cout 0.1 | 0.275 | idem | n/a | n/a |

**Accord** : sur les cas canoniques parapluie et forage, les trois moteurs convergent (EVPI exact = 3.5, EVSI DecInfer-6 exact = 253k). **Désaccord attendu** : DecPyMC-5 cellule 14 utilise une vraisemblance différente (sensi 0.80, spéci 0.90) qui donne EVSI = 23k ; les deux résultats sont cohérents avec leurs senseurs respectifs.

**Limites** : DecInfer-6 et DecPyMC-5 ont chacun 2 exercices dans leur notebook source, pas 3. ICT-12e ajoute 3 exercices stubs C.1 (un par exemple) pour compléter la pédagogie selon [three-exercises-per-notebook.md](../../.claude/rules/three-exercises-per-notebook.md). PyMC MCMC ajoute une variance d'échantillonnage σ ≈ 0.001-0.05 (2000 draws) ; les valeurs closes-form ict.voi sont dans cette barre pour 95% des cas.


## Conclusion — EVPI/EVSI cross-engine pour l'animat incarné

Ce notebook confronte **trois moteurs indépendants** sur les cas canoniques de la valeur de l'information howardienne :

- **EVPI parapluie** = 3.500 (3/3 moteurs : ict.voi + DecInfer-6 + DecPyMC-5) ✓
- **EVSI forage (DecInfer-6 vraisemblance)** = 253_000 (ict.voi exact = DecInfer-6 verbatim) ✓
- **Animat incarné sens proprioceptif** = 10.7% EVPI, discrimination nette (≠ 0% bruit, ≠ 100% oracle) ✓

### Ce qui distingue cette tranche 2/3

**Trois moteurs appelés réellement**, pas un seul. `ict.voi` analytique NumPy donne la référence close-form. PyMC MCMC sampling (`pm.sample`) calcule les posteriors et les EVPI/EVSI par Monte-Carlo, avec variance d'échantillonnage. DecInfer-6 et DecPyMC-5 verbatim published sont les **références cross-engine** que ce notebook reproduit à précision de calcul près.

**Trois exercices stubs C.1** (un par exemple), distribués et exécutables end-to-end selon [three-exercises-per-notebook.md](../../.claude/rules/three-exercises-per-notebook.md) : exercice EVPI à la main, exercice EVSI personnalisé, exercice calibration senseur proprioceptif. Chaque exercice peut être résolu indépendamment, et tous convergent sur le même `ict.voi.evpi` / `evsi` / `evsi_net`.

**L'animat incarné est non dégénéré.** Le sens proprioceptif 85/75 discrimine (10.7% EVPI, ≠ 0% et ≠ 100%) sans tomber dans le cas trivial du senseur parfait ou du bruit pur — c'est la marque de l'incarnation, et le lieu où la valeur howardienne a un sens.

### Suite

- **Tranche 3/3** : extension MCMC (DecPyMC-5 §10) — variance de l'EVSI sous meta-prior.
- **Contrôle cross-kernel runtime** DecInfer-6 (.NET) ↔ PyMC : hors scope de cette tranche (dual-kernel notebook Jupyter complexe), pourrait faire l'objet d'un subagent `.NET` ou d'un notebook dual ultérieur.

See #13569 (tranche 2/3 — trois moteurs cross-engine + trois exercices stubs C.1).
